# Sprint 5 — Streamlit Dashboard

**ITEC631 AI-Driven Network Intrusion Detection System**

This notebook builds and launches an interactive Streamlit dashboard that wraps the full two-stage NIDS pipeline.

| Page | Description |
|------|-------------|
| 🔍 Real-time Detection | Pick a sample → Stage 1 → Stage 2 → SHAP explanation |
| 📊 SHAP Explanations | Pre-computed global SHAP plots from Sprint 4 |
| 📈 Model Performance | F1, accuracy, two-stage vs single-stage comparison |
| 📁 Batch Prediction | Upload CSV → batch predict → download results |

**Required Kaggle inputs:** Sprint 1 output · Sprint 2 output · Sprint 3 output · Sprint 4 output


In [ ]:
import subprocess, sys, os

# install streamlit if not present
try:
    import streamlit
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'streamlit', '-q'], check=True)

import json, pickle
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

print('Setup complete')


## 1 · Path Detection
Locate Sprint 1–4 output files from `/kaggle/input`.


In [ ]:
PROC_DIR     = None   # Sprint 1: test.csv / cicids2017_cleaned.csv
S2_MODEL_DIR = None   # Sprint 2: xgboost.pkl, best_model.json
S3_MODEL_DIR = None   # Sprint 3: stage2_xgboost.pkl, stage2_label_encoder.pkl
S4_OUTPUT_DIR = None  # Sprint 4: shap_stage1_bar.png, sprint4_shap_summary.json

for root, dirs, files in os.walk('/kaggle/input'):
    if 'cicids2017_cleaned.csv' in files or 'test.csv' in files:
        if PROC_DIR is None:
            PROC_DIR = root + '/'
    if 'xgboost.pkl' in files and 'best_model.json' in files:
        S2_MODEL_DIR = root + '/'
    if 'stage2_xgboost.pkl' in files:
        S3_MODEL_DIR = root + '/'
    if 'sprint4_shap_summary.json' in files:
        S4_OUTPUT_DIR = root + '/'

assert PROC_DIR is not None, (
    'test.csv / cicids2017_cleaned.csv not found → add Sprint 1 output as input')
assert S2_MODEL_DIR is not None, (
    'xgboost.pkl not found → add Sprint 2 output as input')
assert S3_MODEL_DIR is not None, (
    'stage2_xgboost.pkl not found → add Sprint 3 output as input')

print(f'PROC_DIR      : {PROC_DIR}')
print(f'S2_MODEL_DIR  : {S2_MODEL_DIR}')
print(f'S3_MODEL_DIR  : {S3_MODEL_DIR}')
print(f'S4_OUTPUT_DIR : {S4_OUTPUT_DIR}  (Sprint 4 SHAP plots)')


## 2 · Verify Assets & Write Config
Check all required files exist, then write a config so the Streamlit process can find them.


In [ ]:
checks = [
    (PROC_DIR,     'test.csv',                      'Sprint 1 — test data'),
    (S2_MODEL_DIR, 'xgboost.pkl',                   'Sprint 2 — Stage 1 model'),
    (S2_MODEL_DIR, 'best_model.json',               'Sprint 2 — model metadata'),
    (S3_MODEL_DIR, 'stage2_xgboost.pkl',            'Sprint 3 — Stage 2 model'),
    (S3_MODEL_DIR, 'stage2_label_encoder.pkl',      'Sprint 3 — label encoder'),
    (S3_MODEL_DIR, 'sprint3_results.json',           'Sprint 3 — results JSON'),
]

all_ok = True
for directory, filename, label in checks:
    path = (directory or '') + filename
    exists = os.path.exists(path)
    if not exists:
        all_ok = False
    print(f"{'✅' if exists else '❌'} {label:40s}  {path}")

# Optional Sprint 4 plots
if S4_OUTPUT_DIR:
    for fname in ['shap_stage1_bar.png', 'shap_stage2_overall.png', 'sprint4_shap_summary.json']:
        p = S4_OUTPUT_DIR + fname
        print(f"{'✅' if os.path.exists(p) else '❌'} Sprint 4 — {fname:35s}  {p}")
else:
    print('⚠️  Sprint 4 output not found — SHAP Explanations page will be limited')

# Write dashboard config so app.py can read it
config = {
    'PROC_DIR':     PROC_DIR,
    'S2_MODEL_DIR': S2_MODEL_DIR,
    'S3_MODEL_DIR': S3_MODEL_DIR,
    'S4_OUTPUT_DIR': S4_OUTPUT_DIR,
}
with open('/kaggle/working/dashboard_config.json', 'w') as f:
    json.dump(config, f, indent=2)
print('\n✅ dashboard_config.json written to /kaggle/working/')


## 3 · Write Streamlit App
`%%writefile` saves `app.py` to `/kaggle/working/`.


In [ ]:
%%writefile /kaggle/working/app.py
# ============================================================
# Sprint 5 — NIDS Streamlit Dashboard
# ITEC631 AI-Driven Network Intrusion Detection System
# ============================================================
import os, json, pickle, io, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import streamlit as st

# ── Page config ──────────────────────────────────────────────
st.set_page_config(
    page_title='NIDS Dashboard',
    page_icon='🛡️',
    layout='wide',
    initial_sidebar_state='expanded',
)

# ── Load dashboard config ─────────────────────────────────────
CONFIG_PATH = '/kaggle/working/dashboard_config.json'
if os.path.exists(CONFIG_PATH):
    with open(CONFIG_PATH) as f:
        _cfg = json.load(f)
    PROC_DIR      = _cfg.get('PROC_DIR')
    S2_MODEL_DIR  = _cfg.get('S2_MODEL_DIR')
    S3_MODEL_DIR  = _cfg.get('S3_MODEL_DIR')
    S4_OUTPUT_DIR = _cfg.get('S4_OUTPUT_DIR')
else:
    # fallback: re-detect
    PROC_DIR = S2_MODEL_DIR = S3_MODEL_DIR = S4_OUTPUT_DIR = None
    for root, dirs, files in os.walk('/kaggle/input'):
        if ('cicids2017_cleaned.csv' in files or 'test.csv' in files) and PROC_DIR is None:
            PROC_DIR = root + '/'
        if 'xgboost.pkl' in files and 'best_model.json' in files:
            S2_MODEL_DIR = root + '/'
        if 'stage2_xgboost.pkl' in files:
            S3_MODEL_DIR = root + '/'
        if 'sprint4_shap_summary.json' in files:
            S4_OUTPUT_DIR = root + '/'

# ── Cached loaders ────────────────────────────────────────────
@st.cache_resource(show_spinner='Loading models...')
def load_models():
    m = {}
    if S2_MODEL_DIR:
        with open(S2_MODEL_DIR + 'xgboost.pkl', 'rb') as f:
            m['stage1'] = pickle.load(f)
        with open(S2_MODEL_DIR + 'best_model.json') as f:
            m['stage1_meta'] = json.load(f)
    if S3_MODEL_DIR:
        with open(S3_MODEL_DIR + 'stage2_xgboost.pkl', 'rb') as f:
            m['stage2'] = pickle.load(f)
        enc_path = S3_MODEL_DIR + 'stage2_label_encoder.pkl'
        if os.path.exists(enc_path):
            with open(enc_path, 'rb') as f:
                m['le2'] = pickle.load(f)
        res_path = S3_MODEL_DIR + 'sprint3_results.json'
        if os.path.exists(res_path):
            with open(res_path) as f:
                m['sprint3_results'] = json.load(f)
    return m

@st.cache_data(show_spinner='Loading test data...')
def load_test_data():
    if not PROC_DIR:
        return None, []
    for fname in ['test.csv', 'cicids2017_cleaned.csv']:
        path = PROC_DIR + fname
        if os.path.exists(path):
            df = pd.read_csv(path, nrows=5000)
            feat_cols = [c for c in df.columns
                         if c not in ('label_binary', 'label_category', 'label_cat_encoded', 'label', 'Label')]
            return df, feat_cols
    return None, []

models       = load_models()
df_test, feature_cols = load_test_data()

attack_classes = ['Botnet', 'BruteForce', 'DoS_DDoS', 'Infiltration', 'PortScan']  # fallback
if 'le2' in models:
    attack_classes = [str(c) for c in models['le2'].classes_ if str(c) != 'nan']

# ── Sidebar ───────────────────────────────────────────────────
st.sidebar.title('🛡️ NIDS Dashboard')
st.sidebar.caption('AI-Driven Network Intrusion Detection')
st.sidebar.markdown('---')

st.sidebar.subheader('System Status')
st.sidebar.write('Stage 1 (Binary):   ' + ('✅ Loaded' if 'stage1' in models else '❌ Missing'))
st.sidebar.write('Stage 2 (Multi):    ' + ('✅ Loaded' if 'stage2' in models else '❌ Missing'))
st.sidebar.write('Label Encoder:      ' + ('✅ Loaded' if 'le2' in models else '❌ Missing'))
st.sidebar.write('SHAP Plots:         ' + ('✅ Available' if S4_OUTPUT_DIR else '⚠️  Missing'))
if df_test is not None:
    st.sidebar.write(f'Test samples:       {len(df_test):,}')
st.sidebar.markdown('---')

PAGE = st.sidebar.radio(
    'Navigate',
    ['🔍 Real-time Detection', '📊 SHAP Explanations',
     '📈 Model Performance', '📁 Batch Prediction'],
)

st.sidebar.markdown('---')
st.sidebar.caption('ITEC631 · Hayoung Kim · 2026')


# ══════════════════════════════════════════════════════════════
# PAGE 1 — Real-time Detection
# ══════════════════════════════════════════════════════════════
if PAGE == '🔍 Real-time Detection':
    st.title('🔍 Real-time Network Intrusion Detection')
    st.markdown('Select a sample from the test dataset and run it through the two-stage pipeline.')

    if df_test is None or 'stage1' not in models:
        st.error('Models or test data not loaded. Check that Sprint 1 and 2 outputs are added as inputs.')
        st.stop()

    col_ctrl, col_info = st.columns([3, 1])
    with col_ctrl:
        if 'label_binary' in df_test.columns:
            label_filter = st.selectbox('Filter by true label', ['All', 'BENIGN only', 'ATTACK only'])
            if label_filter == 'BENIGN only':
                fdf = df_test[df_test['label_binary'] == 0].reset_index(drop=True)
            elif label_filter == 'ATTACK only':
                fdf = df_test[df_test['label_binary'] == 1].reset_index(drop=True)
            else:
                fdf = df_test.reset_index(drop=True)
        else:
            fdf = df_test.reset_index(drop=True)

        sample_idx = st.slider('Sample index', 0, min(len(fdf) - 1, 999), 0)
        sample = fdf.iloc[sample_idx]

    with col_info:
        st.markdown('**Ground truth:**')
        if 'label_binary' in fdf.columns:
            true_bin = sample['label_binary']
            st.markdown(f'Binary: {":green[BENIGN]" if true_bin == 0 else ":red[ATTACK]"}')
        if 'label_category' in fdf.columns:
            st.markdown(f'Category: `{sample["label_category"]}`')

    X_sample = np.array(sample[feature_cols].values, dtype=float).reshape(1, -1)

    if st.button('🚀 Run Detection', type='primary'):
        with st.spinner('Running pipeline...'):
            # Stage 1
            s1_pred  = models['stage1'].predict(X_sample)[0]
            s1_proba = models['stage1'].predict_proba(X_sample)[0]

            st.markdown('---')
            st.subheader('Pipeline Result')
            c1, c2, c3 = st.columns(3)

            with c1:
                if s1_pred == 0:
                    st.success('**BENIGN 🟢**')
                    st.metric('BENIGN confidence', f'{s1_proba[0]*100:.1f}%')
                else:
                    st.error('**ATTACK 🔴**')
                    st.metric('ATTACK confidence', f'{s1_proba[1]*100:.1f}%')

            if s1_pred == 1 and 'stage2' in models:
                s2_pred  = models['stage2'].predict(X_sample)[0]
                s2_proba = models['stage2'].predict_proba(X_sample)[0]
                cls_names = models['le2'].classes_ if 'le2' in models else attack_classes

                pred_label = str(cls_names[s2_pred]) if s2_pred < len(cls_names) else f'Class {s2_pred}'
                with c2:
                    st.warning(f'**{pred_label}**')
                    st.metric('Type confidence', f'{s2_proba[s2_pred]*100:.1f}%')

                with c3:
                    st.markdown('**Attack probabilities:**')
                    top3 = np.argsort(s2_proba)[::-1][:3]
                    for idx in top3:
                        name = str(cls_names[idx]) if idx < len(cls_names) else f'Class {idx}'
                        st.progress(float(s2_proba[idx]),
                                    text=f'{name}: {s2_proba[idx]*100:.1f}%')

            # SHAP bar chart for this sample
            st.markdown('---')
            st.subheader('SHAP Explanation (Stage 1)')
            try:
                import shap
                explainer  = shap.TreeExplainer(models['stage1'])
                shap_vals  = explainer.shap_values(X_sample)
                if isinstance(shap_vals, list):
                    sv = shap_vals[1][0]   # positive class
                elif shap_vals.ndim == 2:
                    sv = shap_vals[0]
                else:
                    sv = shap_vals[0]

                shap_series = pd.Series(sv, index=feature_cols)
                top10 = shap_series.abs().nlargest(10).index
                top10_vals = shap_series[top10]

                fig, ax = plt.subplots(figsize=(10, 4))
                colors = ['#e74c3c' if v > 0 else '#3498db' for v in top10_vals.values]
                ax.barh(top10[::-1], top10_vals.values[::-1], color=colors[::-1])
                ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
                ax.set_xlabel('SHAP Value  (red → ATTACK  |  blue → BENIGN)')
                ax.set_title('Top 10 Features Driving This Prediction')
                plt.tight_layout()
                st.pyplot(fig)
                plt.close(fig)
            except Exception as e:
                st.info(f'SHAP explanation unavailable: {e}')


# ══════════════════════════════════════════════════════════════
# PAGE 2 — SHAP Explanations
# ══════════════════════════════════════════════════════════════
elif PAGE == '📊 SHAP Explanations':
    st.title('📊 SHAP Feature Importance Analysis')
    st.markdown('Global SHAP analysis pre-computed in Sprint 4 — feature importance across the full test set.')

    if not S4_OUTPUT_DIR:
        st.warning('Sprint 4 output not found. Add Sprint 4 output dataset as a notebook input to see SHAP plots.')
    else:
        plot_tabs = st.tabs([
            'Stage 1 — Bar', 'Stage 1 — Beeswarm', 'Stage 1 — Waterfall',
            'Stage 2 — Overall', 'Stage 2 — Heatmap', 'Stage 2 — Per-class',
        ])
        plot_files = [
            'shap_stage1_bar.png', 'shap_stage1_beeswarm.png', 'shap_stage1_waterfall.png',
            'shap_stage2_overall.png', 'shap_stage2_heatmap.png', 'shap_stage2_per_class.png',
        ]
        descriptions = [
            'Mean absolute SHAP value per feature — global Stage 1 importance.',
            'Each dot = one sample. Color = feature value (high/low). Position = SHAP impact.',
            'Individual-level explanation for a BENIGN and an ATTACK sample.',
            'Top 20 features averaged across all attack classes in Stage 2.',
            'Per-class SHAP importance heatmap — which features matter for each attack type.',
            'Individual bar charts for each attack class in Stage 2.',
        ]
        for tab, fname, desc in zip(plot_tabs, plot_files, descriptions):
            with tab:
                st.caption(desc)
                img_path = S4_OUTPUT_DIR + fname
                if os.path.exists(img_path):
                    st.image(img_path, use_container_width=True)
                else:
                    st.warning(f'{fname} not found in Sprint 4 output.')

    # Summary JSON
    if S4_OUTPUT_DIR:
        summary_path = S4_OUTPUT_DIR + 'sprint4_shap_summary.json'
        if os.path.exists(summary_path):
            with open(summary_path) as f:
                summary = json.load(f)
            st.markdown('---')
            st.subheader('Key Findings')
            ka, kb = st.columns(2)
            with ka:
                st.markdown('**Stage 1 — Top 10 Features:**')
                for i, (feat, score) in enumerate(
                    zip(summary.get('stage1_top10', []),
                        summary.get('stage1_top10_scores', [])), 1):
                    st.markdown(f'{i}. `{feat}` — {score:.4f}')
            with kb:
                st.markdown('**Features Shared Across Both Stages:**')
                for feat in summary.get('cross_stage_shared', []):
                    st.markdown(f'• `{feat}`')


# ══════════════════════════════════════════════════════════════
# PAGE 3 — Model Performance
# ══════════════════════════════════════════════════════════════
elif PAGE == '📈 Model Performance':
    st.title('📈 Model Performance Summary')

    s3r = models.get('sprint3_results', {})

    # Stage 1 comparison table
    st.subheader('Stage 1 — Binary Classification (BENIGN vs ATTACK)')
    df_s1 = pd.DataFrame({
        'Model'   : ['Random Forest', 'XGBoost ✅ Best', 'Decision Tree'],
        'F1 Score': [0.9981, 0.9989, 0.9986],
    })
    st.dataframe(df_s1, hide_index=True, use_container_width=True)

    st.markdown('---')

    # Two-stage vs single-stage
    st.subheader('RQ1 — Two-Stage vs Single-Stage Performance')
    pipe_f1   = s3r.get('pipeline_f1',     0.9989)
    single_f1 = s3r.get('single_stage_f1', 0.9980)
    delta_f1  = s3r.get('delta_f1',        round(pipe_f1 - single_f1, 4))
    pipe_acc  = s3r.get('pipeline_acc',    0.0)
    single_acc = s3r.get('single_stage_acc', 0.0)

    mc1, mc2, mc3 = st.columns(3)
    mc1.metric('Two-Stage F1',    f'{pipe_f1:.4f}')
    mc2.metric('Single-Stage F1', f'{single_f1:.4f}', delta=f'{delta_f1:+.4f}', delta_color='normal')
    mc3.metric('Improvement',     f'{delta_f1:+.4f}',
               help='Positive = two-stage is better')

    # Comparison bar chart
    fig, ax = plt.subplots(figsize=(7, 3))
    labels  = ['Single-Stage RF', f'Two-Stage\n({s3r.get("stage1_model","XGB")} + {s3r.get("stage2_model","XGB")})']
    vals    = [single_f1, pipe_f1]
    bars    = ax.barh(labels, vals, color=['#3498db', '#e74c3c'], height=0.5)
    ax.set_xlim(0.995, 1.0005)
    for bar, v in zip(bars, vals):
        ax.text(v + 0.00005, bar.get_y() + bar.get_height()/2,
                f'{v:.4f}', va='center', fontsize=10, fontweight='bold')
    ax.set_xlabel('Weighted F1 Score')
    ax.set_title('Two-Stage vs Single-Stage Comparison')
    plt.tight_layout()
    st.pyplot(fig)
    plt.close(fig)

    st.success(
        f'**RQ1 Answer:** The two-stage pipeline (F1 = {pipe_f1:.4f}) outperforms '
        f'the single-stage approach (F1 = {single_f1:.4f}) by {delta_f1:+.4f}, '
        'confirming that separating binary detection from attack classification improves overall performance.'
    )

    st.markdown('---')
    st.subheader('Stage 2 — Attack Class Distribution')
    atk_classes = s3r.get('stage2_classes', attack_classes)
    st.markdown('Attack classes identified by Stage 2: ' + ', '.join(f'`{c}`' for c in atk_classes))

    st.markdown('---')
    st.subheader('Known Limitations')
    st.warning(
        '**WebAttack missing:** 431 rows were dropped due to a special-character mismatch (◆) '
        'in Sprint 1 CATEGORY_MAP. Documented as a data-quality limitation.\n\n'
        '**Dataset subset:** ~536K of 2.8M rows used due to Kaggle memory constraints.\n\n'
        '**Botnet support discrepancy:** 148 (Stage 2 isolation) vs 68 (full pipeline) '
        'due to NaN fallback routing in the two-stage chain.'
    )


# ══════════════════════════════════════════════════════════════
# PAGE 4 — Batch Prediction
# ══════════════════════════════════════════════════════════════
elif PAGE == '📁 Batch Prediction':
    st.title('📁 Batch Prediction from CSV')
    st.markdown(
        'Upload a CSV with the same features as the CICIDS2017 dataset '
        '(MinMax-scaled, 0–1 range). Label columns are optional.'
    )

    if 'stage1' not in models:
        st.error('Stage 1 model not loaded. Add Sprint 2 output as input.')
        st.stop()

    # Offer sample download from test.csv
    if df_test is not None:
        sample_csv = df_test[feature_cols].head(200).to_csv(index=False)
        st.download_button(
            '⬇️ Download sample CSV (200 rows from test set)',
            data=sample_csv,
            file_name='sample_test_data.csv',
            mime='text/csv',
        )

    uploaded = st.file_uploader('Upload CSV for batch prediction', type=['csv'])

    if uploaded is not None:
        df_up = pd.read_csv(uploaded)
        st.write(f'Uploaded: **{len(df_up):,} rows × {len(df_up.columns)} columns**')
        st.dataframe(df_up.head(5), use_container_width=True)

        matched = [c for c in df_up.columns if c in feature_cols]
        missing = [c for c in feature_cols if c not in df_up.columns]

        st.info(f'Matched {len(matched)}/{len(feature_cols)} expected features. '
                f'Missing features will be filled with 0.')

        if st.button('🚀 Run Batch Detection', type='primary'):
            with st.spinner(f'Running pipeline on {len(df_up):,} samples...'):
                # Build feature matrix (fill missing with 0)
                X_up = np.zeros((len(df_up), len(feature_cols)), dtype=float)
                for j, col in enumerate(feature_cols):
                    if col in df_up.columns:
                        X_up[:, j] = pd.to_numeric(df_up[col], errors='coerce').fillna(0).values

                # Stage 1
                s1_preds = models['stage1'].predict(X_up)
                s1_proba = models['stage1'].predict_proba(X_up)

                results = pd.DataFrame({
                    'stage1_prediction': ['BENIGN' if p == 0 else 'ATTACK' for p in s1_preds],
                    'benign_prob'      : s1_proba[:, 0].round(4),
                    'attack_prob'      : s1_proba[:, 1].round(4),
                    'attack_type'      : 'N/A',
                })

                # Stage 2 for detected attacks
                atk_mask = s1_preds == 1
                if atk_mask.sum() > 0 and 'stage2' in models:
                    s2_preds = models['stage2'].predict(X_up[atk_mask])
                    le2      = models.get('le2')
                    cls_names = le2.classes_ if le2 else attack_classes
                    labels = [str(cls_names[p]) if p < len(cls_names) else 'Unknown'
                              for p in s2_preds]
                    results.loc[atk_mask, 'attack_type'] = labels

            # Summary metrics
            n_total  = len(df_up)
            n_benign = (s1_preds == 0).sum()
            n_attack = atk_mask.sum()

            st.success(f'✅ Done! Detected **{n_attack}** attacks out of **{n_total}** samples.')
            sc1, sc2, sc3 = st.columns(3)
            sc1.metric('Total',   n_total)
            sc2.metric('BENIGN',  n_benign)
            sc3.metric('ATTACK',  n_attack)

            # Attack type breakdown
            if n_attack > 0:
                st.subheader('Attack Type Breakdown')
                atk_counts = results[results['attack_type'] != 'N/A']['attack_type'].value_counts()
                fig, ax = plt.subplots(figsize=(8, 3))
                ax.bar(atk_counts.index, atk_counts.values, color='#e74c3c', edgecolor='white')
                ax.set_ylabel('Count')
                ax.set_title('Detected Attack Types')
                plt.tight_layout()
                st.pyplot(fig)
                plt.close(fig)

            # Preview + download
            st.subheader('Results Preview (first 100 rows)')
            st.dataframe(results.head(100), use_container_width=True)

            out_df  = pd.concat([df_up.reset_index(drop=True), results], axis=1)
            buf     = io.BytesIO()
            out_df.to_csv(buf, index=False)
            st.download_button(
                '⬇️ Download Full Results CSV',
                data=buf.getvalue(),
                file_name='nids_predictions.csv',
                mime='text/csv',
            )


## 4 · Start Streamlit Server
Starts the app in a background thread on port 8501.


In [ ]:
import threading, time

def _run_streamlit():
    subprocess.run(
        ['streamlit', 'run', '/kaggle/working/app.py',
         '--server.port', '8501',
         '--server.headless', 'true',
         '--server.enableCORS', 'false',
         '--server.enableXsrfProtection', 'false'],
        capture_output=True
    )

_t = threading.Thread(target=_run_streamlit, daemon=True)
_t.start()
time.sleep(5)
print('Streamlit running on port 8501')
print('Run the next cell to generate a public URL ↓')


## 5 · Create Public URL
Uses [Cloudflare Tunnel](https://github.com/cloudflare/cloudflared) — no account needed.
The public URL (ending in `.trycloudflare.com`) will appear in the output below.


In [ ]:
import urllib.request, stat

cf_path = '/kaggle/working/cloudflared'
if not os.path.exists(cf_path):
    print('Downloading cloudflared...')
    urllib.request.urlretrieve(
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
        cf_path
    )
    os.chmod(cf_path, stat.S_IRWXU | stat.S_IRWXG | stat.S_IRWXO)
    print('Downloaded.')
else:
    print('cloudflared already present.')

print('Starting tunnel — look for https://....trycloudflare.com in output below:\n')
os.system(f'{cf_path} tunnel --url http://localhost:8501 2>&1')


## 6 · Dashboard Guide

Once the public URL appears above, open it in any browser.

| Page | How to use |
|------|------------|
| 🔍 Real-time Detection | Select a label filter → drag the index slider → click **Run Detection** |
| 📊 SHAP Explanations | Click each tab to explore global SHAP plots from Sprint 4 |
| 📈 Model Performance | View F1 scores and the RQ1 comparison chart |
| 📁 Batch Prediction | Download the sample CSV, modify it, re-upload, click **Run Batch Detection**, then download results |

> **Note:** The Cloudflare tunnel closes when this Kaggle session ends. Re-run cells 5–6 to restart.
